# PredictGuard — Phase 1, Stage 1: Data Understanding & Validation

**Project**: Explainable Predictive Maintenance System  
**Dataset**: Microsoft Azure Predictive Maintenance  
**Author**: PredictGuard Contributors  
**Stage**: 1 of 6 — Data Understanding & Validation

---

## Objective

Before any modelling work begins, we must rigorously understand and validate the raw data.  
This notebook orchestrates the full Stage 1 pipeline. **All reusable logic lives in `src/data_validation.py`** — this notebook is intentionally thin.

### What this notebook does
1. Loads all five PdM CSV files
2. Validates schemas and data types
3. Checks for missing values and duplicate rows
4. Verifies machine ID consistency across all tables
5. Computes telemetry summary statistics
6. Detects telemetry time gaps
7. Detects frozen (stuck) sensors
8. Validates sensor value ranges
9. Verifies cross-table record integrity
10. Generates a complete Data Quality Report (`.md` + `.csv`)
11. Produces 10 publication-quality visualisations

> **Design Decision**: Keeping analysis logic in `src/` rather than inline in notebook cells makes the code importable, testable, and reusable across stages. The notebook acts as a reproducible pipeline runner, not a scratchpad.

---
## 0. Environment Setup & Imports

In [ ]:
import logging
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

# Use inline backend for notebook display
matplotlib.use('Agg')  # will be overridden by %matplotlib inline below

# Make src/ importable when running from notebooks/
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import data_validation as dv

# ---------------------------------------------------------------------------
# Logging configuration
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.StreamHandler(sys.stdout),
    ],
)
logger = logging.getLogger("stage1")
logger.info("Stage 1 pipeline started.")

In [ ]:
# ---------------------------------------------------------------------------
# Paths — adjust DATA_DIR if your CSVs live elsewhere
# ---------------------------------------------------------------------------
DATA_DIR    = project_root / "data" / "raw"
REPORTS_DIR = project_root / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

# Ensure output directories exist
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

logger.info("Data directory  : %s", DATA_DIR)
logger.info("Reports directory: %s", REPORTS_DIR)

---
## 1. Load Datasets

### Dataset Descriptions

| Table | Description | Temporal |
|---|---|---|
| `PdM_telemetry` | Hourly sensor readings: volt, rotate, pressure, vibration | Yes |
| `PdM_errors` | Error codes logged per machine (non-failure anomalies) | Yes |
| `PdM_maint` | Maintenance/component replacement records | Yes |
| `PdM_failures` | Machine failure events per component | Yes |
| `PdM_machines` | Static machine metadata: model, age | No |

> **Design Decision**: `load_all_datasets` parses timestamps at load time using `pd.to_datetime`. Doing this once at ingestion — rather than repeatedly downstream — avoids repeated string-parsing overhead and ensures all timestamp arithmetic is type-safe throughout the pipeline.

In [ ]:
datasets = dv.load_all_datasets(DATA_DIR)

# Convenience aliases
telemetry = datasets["telemetry"]
errors    = datasets["errors"]
maint     = datasets["maint"]
failures  = datasets["failures"]
machines  = datasets["machines"]

logger.info("All datasets loaded successfully.")

In [ ]:
# Quick peek at each table
for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f" Table: {name}  |  shape: {df.shape}")
    print(f"{'='*60}")
    display(df.head(3))
    print("\nDtypes:")
    print(df.dtypes.to_string())

---
## 2. Schema Validation

> **Design Decision**: Schema checks are performed separately from data loading. This allows the pipeline to load data first (fast), then report all schema issues at once rather than failing on the first problem — which is more useful in a production context.

In [ ]:
schema_issues = dv.validate_schemas(datasets)

print("\n── Schema Validation Results ──")
all_ok = True
for table, issues in schema_issues.items():
    if issues:
        all_ok = False
        print(f"  ❌ {table}:")
        for issue in issues:
            print(f"       • {issue}")
    else:
        print(f"  ✅ {table}: OK")

if all_ok:
    print("\n✅ All schemas validated successfully.")

---
## 3. Missing Values & Duplicate Rows

In [ ]:
missing_df = dv.check_missing_values(datasets)

print("\n── Missing Values ──")
missing_flagged = missing_df[missing_df["missing_count"] > 0]
if missing_flagged.empty:
    print("✅ No missing values detected in any table.")
else:
    display(missing_flagged)

In [ ]:
duplicates_df = dv.check_duplicate_rows(datasets)

print("\n── Duplicate Rows ──")
display(duplicates_df)

---
## 4. Machine ID Consistency

> **Design Decision**: Cross-table machine ID verification is critical for joining tables later. Orphan records (events with no machine entry) would corrupt model training. We surface these now, not during feature engineering.

In [ ]:
consistency = dv.verify_machine_consistency(datasets)

print("\n── Machine ID Counts per Table ──")
for t, cnt in consistency["machine_ids_per_table"].items():
    print(f"  {t:<12}: {cnt} unique machine IDs")

print("\n── Orphan Records ──")
print(f"  Orphan errors    : {consistency['orphan_errors']}")
print(f"  Orphan maint     : {consistency['orphan_maint']}")
print(f"  Orphan failures  : {consistency['orphan_failures']}")
print(f"  No telemetry     : {consistency['machines_with_no_telemetry']}")
print(f"  No events        : {len(consistency['machines_with_no_events'])} machines")

---
## 5. Telemetry Summary Statistics

In [ ]:
telemetry_summary = dv.compute_telemetry_summary(telemetry, machines)

print("\n── Overall Telemetry Summary ──")
print(f"  Total machines       : {len(telemetry_summary)}")
print(f"  Total telemetry rows : {telemetry_summary['row_count'].sum():,}")
print(f"  Avg rows/machine     : {telemetry_summary['row_count'].mean():.0f}")
print(f"  Min timestamp        : {telemetry_summary['min_ts'].min()}")
print(f"  Max timestamp        : {telemetry_summary['max_ts'].max()}")
print(f"  Machines < 30 days   : {(telemetry_summary['span_days'] < 30).sum()}")

print("\n── Per-Machine Summary (first 10 rows) ──")
display(telemetry_summary.head(10))

---
## 6. Telemetry Gap Detection

> **Design Decision**: We flag gaps > 3 hours because the telemetry is recorded hourly. A gap > 3 h means at least two consecutive readings are missing — this can confuse lag-based features and rolling aggregations in Stage 2. We record gaps now so that Stage 2 can decide how to handle them (imputation vs. exclusion).

In [ ]:
gaps_df = dv.detect_telemetry_gaps(telemetry, gap_threshold_hours=3.0)

print(f"\n── Telemetry Gaps > 3 hours ──")
print(f"  Total gaps detected      : {len(gaps_df)}")
if not gaps_df.empty:
    print(f"  Machines with gaps       : {gaps_df['machineID'].nunique()}")
    print(f"  Largest gap (hours)      : {gaps_df['gap_hours'].max():.1f}")
    print(f"  Mean gap size (hours)    : {gaps_df['gap_hours'].mean():.2f}")
    print("\n  Top 10 largest gaps:")
    display(gaps_df.head(10))
else:
    print("  ✅ No gaps detected.")

---
## 7. Frozen Sensor Detection

> **Design Decision**: A rolling standard deviation window of 10 readings (= 10 hours) is used. A window of zero std indicates the sensor reported the exact same value ten times consecutively — almost certainly a sensor fault, not a genuine physical reading. The window size is configurable in `dv.DEFAULT_FROZEN_WINDOW`.

In [ ]:
frozen_df = dv.detect_frozen_sensors(telemetry, window=dv.DEFAULT_FROZEN_WINDOW)

print(f"\n── Frozen Sensor Intervals (rolling window = {dv.DEFAULT_FROZEN_WINDOW} readings) ──")
print(f"  Total frozen intervals   : {len(frozen_df)}")
if not frozen_df.empty:
    print(f"  Sensors affected         : {frozen_df['sensor'].unique().tolist()}")
    print(f"  Machines affected        : {frozen_df['machineID'].nunique()}")
    print("\n  Sample frozen intervals:")
    display(frozen_df.head(10))
else:
    print("  ✅ No frozen sensor intervals detected.")

---
## 8. Sensor Range Validation

> **Design Decision**: Out-of-range readings are **flagged, not removed**. Stage 1 is purely diagnostic. Outlier handling strategy (Winsorization, removal, or domain-guided capping) will be decided in Stage 2 based on the findings here. Removing data before understanding it is a common and costly mistake.

In [ ]:
print("\n── Configured Sensor Bounds ──")
for sensor, (lo, hi) in dv.SENSOR_BOUNDS.items():
    print(f"  {sensor:<12}: [{lo}, {hi}]")

telemetry_flagged = dv.validate_sensor_ranges(telemetry, dv.SENSOR_BOUNDS)

flag_cols = [c for c in telemetry_flagged.columns if c.endswith("_out_of_range")]
print("\n── Out-of-Range Sensor Readings ──")
for col in flag_cols:
    sensor = col.replace("_out_of_range", "")
    n = int(telemetry_flagged[col].sum())
    pct = 100.0 * n / len(telemetry_flagged)
    status = "✅" if n == 0 else "⚠️"
    print(f"  {status} {sensor:<12}: {n:>6,} flagged ({pct:.3f}%)")

In [ ]:
# Descriptive statistics for sensor columns
print("\n── Sensor Descriptive Statistics ──")
display(telemetry[dv.SENSOR_COLUMNS].describe().round(3))

---
## 9. Cross-Table Record Integrity

> **Design Decision**: We match event records to telemetry using hour-level granularity (floor to hour). The telemetry is sampled hourly, so any event recorded in the same calendar hour should have matching sensor data. Events with no telemetry match would introduce noise in label generation (Stage 3).

In [ ]:
integrity = dv.verify_cross_table_integrity(datasets)

print("\n── Cross-Table Integrity Check ──")
for key, df in integrity.items():
    status = "✅" if len(df) == 0 else "⚠️"
    print(f"  {status} {key:<30}: {len(df):,} unmatched records")
    if not df.empty:
        display(df.head(5))

---
## 10. Build & Save Data Quality Report

In [ ]:
report_df = dv.build_quality_report(
    datasets=datasets,
    missing_df=missing_df,
    duplicates_df=duplicates_df,
    consistency=consistency,
    gaps_df=gaps_df,
    frozen_df=frozen_df,
    telemetry_flagged=telemetry_flagged,
    integrity=integrity,
    telemetry_summary=telemetry_summary,
)

print(f"\nReport entries: {len(report_df)}")
display(report_df)

In [ ]:
# Save CSV report
dv.save_report_csv(report_df, REPORTS_DIR / "data_quality_report.csv")

# Save Markdown report
dv.save_report_md(
    report_df=report_df,
    datasets=datasets,
    missing_df=missing_df,
    duplicates_df=duplicates_df,
    gaps_df=gaps_df,
    frozen_df=frozen_df,
    telemetry_flagged=telemetry_flagged,
    integrity=integrity,
    telemetry_summary=telemetry_summary,
    consistency=consistency,
    output_path=REPORTS_DIR / "data_quality_report.md",
)

print("\n✅ Reports saved to:")
print(f"   {REPORTS_DIR / 'data_quality_report.csv'}")
print(f"   {REPORTS_DIR / 'data_quality_report.md'}")

---
## 11. Publication-Quality Visualisations

All figures are saved to `reports/figures/`. Each plotting function is self-contained and can be re-run independently.

> **Design Decision**: Figures are generated with `dpi=150` and saved as PNG files for maximum compatibility. All functions use the seaborn `darkgrid` theme for a consistent, professional look across all plots.

In [ ]:
import matplotlib
matplotlib.use('Agg')  # ensure headless save works

logger.info("Generating visualisations …")

### Fig 1 — Missing Value Heatmap

In [ ]:
dv.plot_missing_heatmap(datasets, FIGURES_DIR)
print("✅ Missing value heatmap saved.")

### Fig 2 — Sensor Histograms

In [ ]:
dv.plot_sensor_histograms(telemetry, FIGURES_DIR)
print("✅ Sensor histograms saved.")

### Fig 3 — Sensor Boxplots

In [ ]:
dv.plot_boxplots(telemetry, FIGURES_DIR)
print("✅ Sensor boxplots saved.")

### Fig 4 — Machine Timeline

In [ ]:
dv.plot_machine_timeline(telemetry, failures, FIGURES_DIR, max_machines=20)
print("✅ Machine timeline saved.")

### Fig 5 — Records per Machine

In [ ]:
dv.plot_records_per_machine(datasets, FIGURES_DIR)
print("✅ Records per machine saved.")

### Fig 6 — Telemetry Gap Distribution

In [ ]:
dv.plot_gap_distribution(gaps_df, FIGURES_DIR)
print("✅ Gap distribution saved.")

### Fig 7 — Sensor Correlation Heatmap

In [ ]:
dv.plot_sensor_correlation(telemetry, FIGURES_DIR)
print("✅ Sensor correlation heatmap saved.")

### Fig 8 — Failure Count per Component

In [ ]:
dv.plot_failure_counts(failures, FIGURES_DIR)
print("✅ Failure counts saved.")

### Fig 9 — Maintenance Frequency

In [ ]:
dv.plot_maintenance_frequency(maint, FIGURES_DIR)
print("✅ Maintenance frequency saved.")

### Fig 10 — Error Frequency

In [ ]:
dv.plot_error_frequency(errors, FIGURES_DIR)
print("✅ Error frequency saved.")

---
## 12. Stage 1 Summary & Conclusions

### Key Findings

Run the notebook to populate these findings automatically.

### Stage 1 Checklist

| Check | Status |
|---|---|
| All 5 CSV files loaded | ✅ |
| Timestamps parsed correctly | ✅ |
| Schema validated | ✅ |
| Missing values checked | ✅ |
| Duplicate rows detected | ✅ |
| Machine ID consistency verified | ✅ |
| Telemetry summary computed | ✅ |
| Telemetry gaps detected | ✅ |
| Frozen sensors detected | ✅ |
| Sensor ranges validated | ✅ |
| Cross-table integrity verified | ✅ |
| Data quality report saved | ✅ |
| 10 visualisations generated | ✅ |

### Next Steps — Stage 2: Feature Engineering

With a clean understanding of the data, Stage 2 will:
- Create rolling-window lag features from telemetry
- Encode error and maintenance history
- Build a unified feature matrix per machine per hour
- Handle the gaps and outliers identified in this stage

In [ ]:
logger.info("Stage 1 pipeline completed successfully.")
print("\n" + "="*60)
print(" PredictGuard — Stage 1 COMPLETE")
print("="*60)
print(f"  Reports  → {REPORTS_DIR}")
print(f"  Figures  → {FIGURES_DIR}")
print("="*60)